# Week 1 Practical: Exploring Molecules with RDKit
**AI for Drug Discovery — A Neuroscience Perspective**

> **Instructor:** Dr. Étienne Serbe-Kamp — computational neuroscientist studying visual motion detection circuits in *Drosophila* (fruit flies). His research spans electron-microscopy connectomics ([Shinomiya, Serbe-Kamp et al., *Neuron* 2016](https://doi.org/10.1016/j.neuron.2019.02.043)), multilevel motion opponency via GluClα-mediated inhibition ([Serbe-Kamp et al., *Nature Neuroscience* 2023](https://doi.org/10.1038/s41593-023-01443-z)), and directional tuning maps of elementary motion detectors ([Serbe et al., *Nature* 2013](https://doi.org/10.1038/nature12320)). He is Co-Director of Summer Fellowships at **Backyard Brains**, where students use SpikerBot/SpikerBox devices and citizen-science platforms (plant electrophysiology, the ERGo! electroretinogram platform) to explore real neural signals.

🧠 **Why neuroscience meets drug discovery:** Many of the most important drug targets — ion channels, neurotransmitter receptors, and synaptic transporters — were first characterised by neuroscientists. This course bridges **computational neuroscience** (modelling neural circuits, connectomics, electrophysiology) with **AI-driven drug discovery** (molecular representations, property prediction, generative chemistry). The molecules you will analyse today include neurotransmitters, neuroactive drugs, and ion-channel modulators that sit at this intersection.

Welcome to the first practical session of the AI for Drug Discovery course! In this hands-on notebook, you will learn the foundational computational chemistry skills that underpin all modern drug discovery pipelines.

**Why does this matter?** Drug discovery is one of the most expensive and time-consuming endeavors in science. On average, it takes **10-15 years** and **$1-2 billion** to bring a single drug from initial concept to market approval. Computational methods — from simple molecular property filters to state-of-the-art deep learning — can dramatically accelerate this process by identifying promising molecules *in silico* before expensive laboratory experiments.

**What you will learn today:**
1. **Install and import RDKit** — the industry-standard open-source cheminformatics toolkit used by virtually every pharmaceutical company and academic drug discovery group worldwide
2. **Load a real molecular dataset** — working with the Delaney (ESOL) solubility dataset, a classic benchmark in cheminformatics
3. **Parse SMILES and visualize molecules** — converting text-based molecular representations into 2D structural drawings
4. **Calculate molecular properties** — computing physicochemical descriptors (MW, LogP, TPSA, etc.) that predict drug behavior
5. **Plot property distributions** — using statistical visualization to understand your chemical dataset
6. **Apply Lipinski's Rule of Five** — the most famous drug-likeness filter in pharmaceutical science
7. **Compute molecular similarity** — using fingerprints to quantify how structurally similar two molecules are
8. **Simulate a neuron** — run a Hodgkin-Huxley model and explore how drugs alter ion-channel conductances

**Prerequisites:** Basic Python programming. No chemistry or neuroscience background is assumed — all concepts will be explained from first principles.

## 1. Setup and Installation

Run the cell below to install RDKit in Google Colab. If you're running locally, use `pip install rdkit-pypi`.

**What is RDKit?** RDKit is an open-source cheminformatics library originally developed by Greg Landrum while at the Rational Drug Design group at Novartis (Basel, Switzerland) in the early 2000s. It has since grown into the most widely used open-source cheminformatics toolkit in the world, with contributions from hundreds of developers across academia and industry. RDKit is written primarily in C++ for performance, with Python bindings that make it accessible for data science workflows.

**Key capabilities of RDKit include:**
- Reading/writing molecules in various formats (SMILES, SDF, MOL, SMARTS, InChI)
- Substructure and similarity searching
- Molecular descriptor calculation (200+ descriptors)
- 2D depiction and 3D coordinate generation
- Chemical reaction handling and retrosynthesis
- Fingerprint generation (Morgan/ECFP, MACCS, topological, etc.)
- Force field optimization (MMFF94, UFF)
- Integration with machine learning libraries (scikit-learn, PyTorch, TensorFlow)

**Why not commercial software?** While commercial packages like Schrödinger, OpenEye, and Chemical Computing Group offer powerful tools, RDKit's open-source nature makes it the standard for reproducible research and is used in production at companies like Google, Microsoft, Novartis, Roche, and many biotech startups.

In [ ]:
# Install RDKit and dependencies in Google Colab
# !pip install installs Python packages from PyPI (Python Package Index)
# rdkit-pypi: The RDKit cheminformatics library, packaged for easy pip installation.
#   RDKit was originally developed by Greg Landrum at Novartis in the early 2000s and
#   is now the most widely used open-source toolkit for cheminformatics and computational
#   chemistry. It provides functions for molecular I/O, substructure searching, fingerprinting,
#   descriptor calculation, 2D/3D coordinate generation, and much more.
# pandas: The fundamental data analysis library for Python, providing DataFrame structures
#   that we use to store and manipulate molecular datasets. Created by Wes McKinney in 2008.
# matplotlib: The foundational plotting library for Python, created by John Hunter in 2003.
#   We use it to create histograms, scatter plots, and other visualizations of molecular data.
# seaborn: A statistical visualization library built on top of matplotlib that provides
#   higher-level, more attractive default plots. Created by Michael Waskom.
# The -q flag suppresses verbose installation output for a cleaner notebook experience.
!pip install rdkit-pypi pandas matplotlib seaborn -q

# Import the warnings module from the Python standard library.
# This module allows us to control how Python warnings are displayed.
import warnings

# Suppress all warning messages to keep the notebook output clean.
# RDKit and other scientific libraries sometimes emit non-critical deprecation
# or runtime warnings that can clutter the output and confuse students.
# 'ignore' tells Python to silently discard all warnings.
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# CORE IMPORTS FOR CHEMINFORMATICS
# These libraries form the foundation of computational chemistry
# workflows used in pharma and biotech companies worldwide.
# ============================================================

# Import the Chem module from RDKit — this is the core module of the RDKit
# cheminformatics library. It provides the fundamental Mol object and functions
# for reading/writing molecules in SMILES, SDF, MOL, and other formats.
# RDKit was created by Greg Landrum at Novartis and is the most widely used
# open-source cheminformatics toolkit (https://www.rdkit.org).
from rdkit import Chem

# Draw: provides functions to render 2D images of molecules (MolToImage, MolsToGridImage).
# Descriptors: provides 200+ molecular descriptor calculators (MW, LogP, TPSA, etc.).
# AllChem: provides advanced chemistry functions including conformer generation,
#   force field minimization, and Morgan fingerprint generation.
# Lipinski: provides Lipinski Rule-of-Five descriptor calculators specifically.
from rdkit.Chem import Draw, Descriptors, AllChem, Lipinski

# PandasTools integrates RDKit molecule objects directly into pandas DataFrames,
# allowing molecules to be displayed as images within notebook tables.
from rdkit.Chem import PandasTools

# pandas (imported as pd by convention) — the cornerstone data manipulation library.
# DataFrames provide SQL-like operations (filter, group, join) on tabular data.
# Nearly every data science and cheminformatics workflow in Python uses pandas.
import pandas as pd

# NumPy (imported as np by convention) — the fundamental numerical computing library.
# Provides efficient N-dimensional arrays and mathematical operations.
# It underpins virtually all scientific computing in Python.
import numpy as np

# matplotlib.pyplot (imported as plt by convention) — the core plotting library.
# Provides MATLAB-like plotting functions: plot(), scatter(), hist(), etc.
import matplotlib.pyplot as plt

# seaborn (imported as sns by convention) — statistical data visualization.
# Built on matplotlib, it provides higher-level functions for attractive plots.
# The name "sns" is a reference to Samuel Norman Seaborn from The West Wing.
import seaborn as sns

# Set the default visual style for all seaborn/matplotlib plots in this notebook.
# 'whitegrid' adds subtle gridlines on a white background, making it easy to
# read values from plots. Other options: 'darkgrid', 'white', 'dark', 'ticks'.
sns.set_style('whitegrid')

# Print the installed RDKit version to confirm the library loaded correctly.
# This is a good practice for reproducibility — if code breaks in the future,
# knowing which version was used helps diagnose compatibility issues.
print('RDKit version:', Chem.rdBase.rdkitVersion)

# Confirm that all imports succeeded without errors.
print('Setup complete!')

## 2. Your First Molecule

Let's start by parsing a SMILES string and visualizing a molecule.

### What is SMILES?

**SMILES** (Simplified Molecular Input Line Entry System) is a line notation for representing chemical structures as text strings. It was invented by **David Weininger** in 1986-1988 while working at the US Environmental Protection Agency (EPA) in Duluth, Minnesota. Weininger went on to found Daylight Chemical Information Systems, which became hugely influential in cheminformatics.

**Key SMILES rules:**
- **Atoms** are represented by their atomic symbols: `C` (carbon), `N` (nitrogen), `O` (oxygen), `S` (sulfur), etc.
- **Single bonds** are implicit (atoms written next to each other are bonded): `CC` = ethane (C-C)
- **Double bonds** use `=`: `C=O` = formaldehyde
- **Triple bonds** use `#`: `C#N` = hydrogen cyanide
- **Branches** use parentheses: `CC(=O)O` = acetic acid (methyl group, then a branch with =O, then O)
- **Rings** use matching numbers: `C1CCCCC1` = cyclohexane (atoms 1 connects back to atom 1)
- **Aromatic atoms** are lowercase: `c1ccccc1` = benzene (aromatic carbons)
- **Stereochemistry**: `@`/`@@` for chiral centers, `/`/`\` for E/Z double bonds
- **Charges**: `[NH4+]` = ammonium ion, `[O-]` = oxide
- **Isotopes**: `[13C]` = carbon-13

**Why SMILES matters for AI/ML:** SMILES strings can be treated as a "molecular language," enabling the use of natural language processing (NLP) techniques for molecular generation and property prediction. Models like SMILES-based variational autoencoders (VAEs) and transformer models operate directly on SMILES text.

**Canonical SMILES:** There are many valid SMILES for the same molecule (e.g., `OCC` and `CCO` both represent ethanol). RDKit generates **canonical SMILES** — a unique, deterministic string for each molecule — enabling exact molecular comparisons and database lookups.

**SMILES limitations:** SMILES cannot represent some complex features like coordination bonds in metal complexes, and long SMILES strings for large molecules can be error-prone. Alternatives include **InChI** (IUPAC's canonical identifier), **SELFIES** (which guarantees chemical validity), and **molecular graphs** (used in graph neural networks).

**Reference:** Weininger, D. (1988). SMILES, a chemical language and information system. 1. Introduction to methodology and encoding rules. *J. Chem. Inf. Comput. Sci.* 28:31-36.

---
🧠 **Neuroscience connection:** Many of the most successful drugs in history target proteins first discovered by neuroscientists. Ion channels (the Na⁺/K⁺ channels in the Hodgkin-Huxley model, GluCl channels in *Drosophila*), neurotransmitter receptors (GABA-A, serotonin 5-HT, dopamine D2, nicotinic acetylcholine), and synaptic transporters (SERT, DAT, NET) are all encoded by SMILES strings just like the small-molecule drugs that bind to them. Throughout this notebook, we will visualise both neurotransmitters and the drugs that modulate them.

In [ ]:
# ============================================================
# PARSING AND VISUALIZING YOUR FIRST MOLECULE: ASPIRIN
# This cell demonstrates the fundamental workflow of computational
# chemistry: text representation -> molecular object -> properties.
# ============================================================

# Define the SMILES string for aspirin (acetylsalicylic acid).
# SMILES stands for Simplified Molecular Input Line Entry System.
# It was invented by David Weininger in 1986-1988 at the US EPA.
# In SMILES notation:
#   CC(=O)  = an acetyl group (methyl + carbonyl)
#   O       = an ester oxygen connecting the acetyl to the ring
#   c1ccccc1 = a benzene ring (lowercase 'c' = aromatic carbon)
#   C(=O)O  = a carboxylic acid group
# Aspirin was first synthesized by Felix Hoffmann at Bayer in 1897
# and remains one of the world's most widely used drugs.
aspirin_smiles = 'CC(=O)Oc1ccccc1C(=O)O'

# Chem.MolFromSmiles() parses the SMILES string and creates an RDKit Mol object.
# The Mol object stores the full molecular graph: atoms, bonds, stereochemistry,
# aromaticity, and ring information. If the SMILES is invalid, this returns None.
# Internally, RDKit: (1) tokenizes the SMILES, (2) builds the molecular graph,
# (3) perceives aromaticity, (4) assigns implicit hydrogens.
aspirin = Chem.MolFromSmiles(aspirin_smiles)

# Display the SMILES string — this is the canonical text representation.
# Canonical SMILES is a unique string for each molecule, enabling exact lookups.
print(f'SMILES: {aspirin_smiles}')

# CalcMolFormula() computes the molecular formula (e.g., C9H8O4 for aspirin).
# It counts all atoms (including implicit hydrogens) and returns a string.
print(f'Molecular Formula: {Chem.rdMolDescriptors.CalcMolFormula(aspirin)}')

# Descriptors.MolWt() calculates the average molecular weight in Daltons (Da).
# It uses average atomic weights (accounting for natural isotope distributions).
# Aspirin's MW is ~180.16 Da — well within the "drug-like" range of 150-500 Da.
print(f'Molecular Weight: {Descriptors.MolWt(aspirin):.2f} Da')

# Draw.MolToImage() generates a 2D depiction of the molecule as a PIL Image.
# RDKit uses the Coordgen library (from Schrodinger) or its own coordinate
# generator to compute aesthetically pleasing 2D layouts.
# size=(400, 300) sets the output image dimensions in pixels.
Draw.MolToImage(aspirin, size=(400, 300))

In [ ]:
# ============================================================
# VISUALIZING MULTIPLE WELL-KNOWN DRUGS
# This cell demonstrates batch molecule visualization — a common
# task when comparing chemical series or reviewing hit lists.
# ============================================================

# Define a dictionary mapping drug names to their SMILES strings.
# Each of these drugs represents a different therapeutic area and mechanism:
# - Aspirin: COX-1/COX-2 inhibitor (anti-inflammatory, pain)
# - Caffeine: adenosine receptor antagonist (stimulant)
# - Ibuprofen: COX inhibitor (NSAID, pain/inflammation)
# - Penicillin V: beta-lactam antibiotic (targets bacterial cell wall synthesis)
# - Paracetamol (acetaminophen): mechanism still debated! Likely COX-3/TRPA1
# - Diazepam (Valium): GABA-A receptor positive allosteric modulator (anxiolytic)
#
# --- NEUROSCIENCE-RELEVANT MOLECULES ---
# - Serotonin (5-HT): monoamine neurotransmitter; targets include 5-HT1A-7
#   receptors. SSRIs (e.g. fluoxetine) block its reuptake transporter SERT.
# - Dopamine: catecholamine neurotransmitter central to reward, movement,
#   and motivation. D2 receptor is the primary target of antipsychotics.
# - GABA (gamma-aminobutyric acid): the main inhibitory neurotransmitter
#   in the mammalian CNS. GABA-A receptors are Cl- channels — the target
#   of benzodiazepines, barbiturates, and anaesthetics.
# - Ivermectin: macrocyclic lactone antiparasitic that activates GluCl
#   (glutamate-gated chloride) channels in invertebrates. GluClα is a key
#   receptor studied in Drosophila motion vision circuits (Serbe-Kamp et al.,
#   Nature Neuroscience 2023). Nobel Prize in Physiology or Medicine 2015
#   (William Campbell & Satoshi Ōmura).
drug_dict = {
    'Aspirin': 'CC(=O)Oc1ccccc1C(=O)O',
    'Caffeine': 'Cn1c(=O)c2c(ncn2C)n(C)c1=O',
    'Ibuprofen': 'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
    'Penicillin V': 'CC1(C)S[C@@H]2[C@H](NC(=O)COc3ccccc3)C(=O)N2[C@@H]1C(=O)O',
    'Paracetamol': 'CC(=O)Nc1ccc(O)cc1',
    'Diazepam': 'CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21',
    # --- Neurotransmitters & neuroactive drugs ---
    'Serotonin (5-HT)': 'NCCc1c[nH]c2ccc(O)cc12',     # indoleamine neurotransmitter
    'Dopamine': 'NCCc1ccc(O)c(O)c1',                    # catecholamine neurotransmitter
    'GABA': 'NCCCC(=O)O',                                # chief inhibitory NT in CNS
    'Ivermectin': 'CC(C(=O)[C@@H]1C[C@H](C/C=C/[C@@H]2C[C@@H](C[C@@H]3CC=CC(=O)[C@@H]3/C=C/[C@@H]2OC)OC4OC(C)[C@@H](OC5OC(C)[C@@H](O)[C@H](OC)C5)[C@H](O)C4)O[C@]6(C[C@@H](OC)[C@H](O)[C@H](C)O6)C)C',
}

# Convert each SMILES string to an RDKit Mol object using a list comprehension.
# Chem.MolFromSmiles() is called for each SMILES value in the dictionary.
# The resulting list of Mol objects is needed by the grid drawing function.
mols = [Chem.MolFromSmiles(smi) for smi in drug_dict.values()]

# Extract the drug names to use as labels below each molecule in the grid image.
legends = list(drug_dict.keys())

# Draw.MolsToGridImage() creates a grid of 2D molecular depictions.
# molsPerRow=4: arrange molecules in rows of 4 to accommodate 10 molecules.
# subImgSize=(400, 300): each individual molecule image is 400x300 pixels.
# legends=legends: display the drug name below each molecule structure.
# This function returns a PIL Image that Jupyter automatically displays.
img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(400, 300), legends=legends)

# Display the grid image inline in the notebook.
# In Jupyter, simply writing a variable name as the last line of a cell
# triggers the notebook's display mechanism (calls _repr_png_ for images).
img

## 3. Load a Real Dataset: EGFR Inhibitors

We'll use a curated dataset of molecules with measured aqueous solubility. Originally, this section targeted EGFR (Epidermal Growth Factor Receptor) inhibitors — EGFR is a major cancer drug target, and drugs like **gefitinib** (Iressa), **erlotinib** (Tarceva), and **osimertinib** (Tagrisso) are EGFR tyrosine kinase inhibitors used to treat non-small cell lung cancer.

For this practical, we use the **Delaney (ESOL) solubility dataset**, one of the most famous benchmark datasets in cheminformatics. It was published by John Delaney in 2004 and contains ~1,128 small organic molecules with experimentally measured aqueous solubility values.

### Why is solubility important?
Aqueous solubility is one of the most critical properties in drug development:
- A drug must **dissolve** in gastrointestinal fluids to be absorbed orally
- Poor solubility is the **#1 reason** drug candidates fail in development
- The **Biopharmaceutics Classification System (BCS)** classifies drugs by solubility and permeability
- ~40% of marketed drugs and ~90% of drug candidates in development have poor water solubility

### What is ChEMBL?
**ChEMBL** is a manually curated database of bioactive molecules with drug-like properties, maintained by the European Bioinformatics Institute (EMBL-EBI). It contains over **2 million compounds** and **20 million bioactivity measurements** extracted from the medicinal chemistry literature. It is the primary data source for most AI drug discovery research.

**Reference:** Gaulton, A. et al. (2017). The ChEMBL database in 2017. *Nucleic Acids Research* 45:D986-D994.

In [ ]:
# ============================================================
# LOADING A REAL MOLECULAR DATASET: DELANEY SOLUBILITY
# The ESOL (Estimated SOLubility) dataset was published by John Delaney
# in 2004 (J. Chem. Inf. Comput. Sci. 44:1000-1005). It contains
# ~1128 small organic molecules with experimentally measured aqueous
# solubility values. This is one of the most widely used benchmark
# datasets in cheminformatics and ML for drug discovery.
# Aqueous solubility is critical: a drug must dissolve in body fluids
# to be absorbed and reach its target.
# ============================================================

# Attempt to download the dataset from the DeepChem GitHub repository.
# DeepChem is an open-source library for deep learning in chemistry,
# maintained by Bharath Ramsundar and contributors.
try:
    # Construct the URL pointing to the raw CSV file on GitHub.
    # This is the processed version of the Delaney ESOL dataset that
    # includes pre-computed molecular descriptors alongside SMILES.
    url = 'https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv'

    # pd.read_csv() downloads and parses the CSV file into a DataFrame.
    # Each row is one molecule; columns include SMILES, measured solubility,
    # and several pre-computed molecular descriptors.
    df = pd.read_csv(url)

    # Report how many molecules were loaded and what columns are available.
    # This is good practice to verify the data loaded correctly.
    print(f'Loaded {len(df)} molecules from Delaney solubility dataset')
    print(f'Columns: {list(df.columns)}')

    # Display the first 5 rows of the DataFrame as a formatted table.
    # .head() is a pandas method that returns the first n rows (default=5).
    df.head()

# If the download fails (e.g., no internet, URL changed), use a fallback dataset.
except Exception as e:
    # Print the error message so students know what went wrong.
    print(f'Could not download dataset: {e}')

    # Create a small fallback dataset with 10 common organic molecules.
    # These SMILES represent: ethanol, benzene, acetic acid, octane, glycerol,
    # naphthalene, isopropanol, undecane, benzyl alcohol, and aspirin.
    # The solubility values are approximate experimentally measured values
    # in units of log(mol/L).
    data = {
        'smiles': ['CCO', 'c1ccccc1', 'CC(=O)O', 'CCCCCCCC', 'OCC(O)CO',
                   'c1ccc2ccccc2c1', 'CC(C)O', 'CCCCCCCCCCC', 'OCc1ccccc1', 'CC(=O)Oc1ccccc1C(=O)O'],
        'measured log solubility in mols per litre': [-0.77, -0.77, 1.15, -3.26, 0.55,
                                                      -2.04, 0.25, -4.23, -0.54, -1.63]
    }

    # Create a pandas DataFrame from the dictionary.
    # Keys become column names; values become column data.
    df = pd.DataFrame(data)

    # Report that we are using the smaller fallback dataset.
    print(f'Using fallback dataset with {len(df)} molecules')

In [ ]:
# ============================================================
# PARSE SMILES AND VALIDATE MOLECULES
# Not all SMILES strings are valid or parseable. This cell converts
# text representations to molecular objects and filters out failures.
# ============================================================

# Apply Chem.MolFromSmiles to every SMILES string in the 'smiles' column.
# .apply() runs a function on each element of a pandas Series.
# lambda s: Chem.MolFromSmiles(s) is an anonymous function that takes a
# SMILES string 's' and returns an RDKit Mol object (or None if invalid).
# The result is stored in a new column 'mol' in the DataFrame.
df['mol'] = df['smiles'].apply(lambda s: Chem.MolFromSmiles(s))

# Check which molecules were successfully parsed.
# .notna() returns True for non-None values, False for None values.
# A None value means RDKit could not parse the SMILES (syntax error,
# impossible valence, etc.). In real datasets, 1-5% of SMILES may fail.
valid = df['mol'].notna()

# Report how many molecules passed validation out of the total.
# This gives students a sense of data quality.
print(f'Valid molecules: {valid.sum()} / {len(df)}')

# Filter the DataFrame to keep only rows with valid Mol objects.
# reset_index(drop=True) re-numbers the rows from 0 to N-1, discarding
# the old index. This prevents index gaps after filtering.
df = df[valid].reset_index(drop=True)

# Display a grid of the first 12 molecules for visual inspection.
# .head(12) selects the first 12 rows; .tolist() converts to a Python list.
mols_sample = df['mol'].head(12).tolist()

# Draw.MolsToGridImage creates a grid image of multiple molecules.
# molsPerRow=4: show 4 molecules per row (so 12 molecules = 3 rows).
# subImgSize=(300, 250): each molecule image is 300x250 pixels.
Draw.MolsToGridImage(mols_sample, molsPerRow=4, subImgSize=(300, 250))

## 4. Calculate Molecular Properties

RDKit can compute hundreds of molecular descriptors. Let's calculate the most important ones for drug discovery.

### What are molecular descriptors?
**Molecular descriptors** are numerical values that encode structural, physical, chemical, or topological information about a molecule. They are the "features" that machine learning models use to predict molecular properties and biological activities. The concept dates back to the 1960s with Hansch and Fujita's work on quantitative structure-activity relationships (QSAR).

### Key descriptors we'll calculate:
| Descriptor | What it measures | Why it matters for drugs |
|-----------|-----------------|------------------------|
| **Molecular Weight (MW)** | Total mass of the molecule | Oral drugs typically 150-500 Da; larger molecules have poorer gut absorption |
| **LogP** | Lipophilicity (oil/water partitioning) | Affects membrane permeation, solubility, protein binding; ideal range 1-3 |
| **HBD** | Hydrogen bond donors (NH, OH groups) | Too many reduce membrane permeation; Lipinski: ≤5 |
| **HBA** | Hydrogen bond acceptors (N, O atoms) | Too many reduce membrane permeation; Lipinski: ≤10 |
| **TPSA** | Topological polar surface area | Predicts oral absorption (<140 Å²) and BBB penetration (<90 Å²) |
| **Rotatable Bonds** | Molecular flexibility | Affects oral bioavailability (Veber: ≤10) and binding entropy cost |
| **Aromatic Rings** | Flat ring systems | π-stacking with proteins; too many → solubility/metabolism issues |
| **Heavy Atoms** | Non-hydrogen atom count | Proxy for size; used in ligand efficiency calculations |

### The concept of "drug-likeness"
Not every molecule that binds a target can become a drug. It must also be **absorbed** from the gut, **distributed** through the body, **metabolized** at a reasonable rate, and **excreted** safely (collectively known as **ADME**). The descriptors above are simple proxies for these complex pharmacokinetic processes.

**Reference:** Ertl, P. et al. (2000). Fast calculation of molecular polar surface area as a sum of fragment-based contributions. *J. Med. Chem.* 43:3714-3717.

In [ ]:
# ============================================================
# CALCULATE KEY MOLECULAR DESCRIPTORS
# Molecular descriptors are numerical values that characterize a
# molecule's physical, chemical, or structural properties. They are
# the features used in QSAR (Quantitative Structure-Activity
# Relationship) models and machine learning for drug discovery.
# ============================================================

# MW (Molecular Weight) in Daltons (g/mol).
# Drugs typically have MW between 150-500 Da (Lipinski's rule).
# Larger molecules have poorer oral absorption because they
# can't easily cross cell membranes by passive diffusion.
df['MW'] = df['mol'].apply(Descriptors.MolWt)

# LogP (octanol-water partition coefficient, logarithmic).
# Measures lipophilicity — how much a molecule prefers oil over water.
# Calculated using Wildman-Crippen method (atom-based contributions).
# LogP 1-3 is ideal for oral drugs; too high = poor solubility, too low = poor membrane permeation.
df['LogP'] = df['mol'].apply(Descriptors.MolLogP)

# HBD (Hydrogen Bond Donors) — count of NH and OH groups.
# These form hydrogen bonds with water, affecting solubility and
# membrane permeation. Lipinski's rule: HBD <= 5.
df['HBD'] = df['mol'].apply(Descriptors.NumHDonors)

# HBA (Hydrogen Bond Acceptors) — count of N and O atoms.
# These accept hydrogen bonds. Lipinski's rule: HBA <= 10.
# Too many H-bond donors/acceptors = high desolvation penalty = poor absorption.
df['HBA'] = df['mol'].apply(Descriptors.NumHAcceptors)

# TPSA (Topological Polar Surface Area) in Angstroms squared.
# Sum of surface area contributed by polar atoms (N, O, and their H atoms).
# TPSA < 140 Å² correlates with good oral absorption.
# TPSA < 90 Å² correlates with good blood-brain barrier penetration.
# Developed by Ertl et al. (2000), J. Med. Chem. 43:3714-3717.
df['TPSA'] = df['mol'].apply(Descriptors.TPSA)

# RotBonds (Number of Rotatable Bonds).
# Counts bonds that can freely rotate (excluding terminal bonds, double bonds,
# and bonds in rings). More rotatable bonds = more flexibility.
# Veber's rule: RotBonds <= 10 for good oral bioavailability.
# Too many = entropic penalty upon binding to target protein.
df['RotBonds'] = df['mol'].apply(Descriptors.NumRotatableBonds)

# AromaticRings — count of aromatic ring systems.
# Aromatic rings contribute to pi-pi stacking interactions with protein targets
# but also increase planarity, which can cause problems with crystallization
# and metabolic stability (CYP-mediated oxidation).
df['AromaticRings'] = df['mol'].apply(Descriptors.NumAromaticRings)

# HeavyAtoms — count of non-hydrogen atoms.
# A rough proxy for molecular size. Typical drugs have 20-40 heavy atoms.
# Ligand efficiency = binding energy / heavy atom count — a key medicinal
# chemistry metric for comparing potency across different-sized molecules.
df['HeavyAtoms'] = df['mol'].apply(Descriptors.HeavyAtomCount)

# Confirm that all descriptor calculations completed successfully.
print('Molecular properties calculated!')

# Display summary statistics (count, mean, std, min, 25%, 50%, 75%, max)
# for the key descriptors. .describe() provides a statistical overview.
# .round(2) rounds all values to 2 decimal places for readability.
df[['smiles', 'MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds']].describe().round(2)

## 5. Visualize Property Distributions

Understanding the distribution of molecular properties in your dataset is essential for:
1. **Quality control** — identifying data errors, outliers, or unexpected patterns
2. **Chemical space analysis** — understanding what types of molecules your dataset contains
3. **Comparison with drug-like space** — checking if your molecules fall within known drug-like ranges
4. **Feature engineering** — deciding which properties need normalization or transformation for ML models

The histograms below show the distribution of each key property across all molecules in the dataset. The red dashed line marks the median (50th percentile) — half the molecules fall on each side.

**What to look for:**
- **MW distribution:** Most approved oral drugs have MW between 200-500 Da. A dataset skewed toward very high MW may contain "undruggable" molecules.
- **LogP distribution:** The ideal range is roughly 1-3 for oral drugs. Very negative LogP = too hydrophilic (poor membrane permeation); very positive LogP = too lipophilic (poor solubility, high metabolic clearance).
- **Skewness:** Right-skewed distributions (long tail to the right) are common for MW and TPSA in diverse chemical libraries.

In [ ]:
# ============================================================
# VISUALIZE PROPERTY DISTRIBUTIONS
# Histograms reveal the distribution of molecular properties in our
# dataset. Understanding these distributions helps identify outliers,
# assess drug-likeness, and compare with known drug property ranges.
# ============================================================

# Create a 2x3 grid of subplots (6 panels total, one per property).
# figsize=(15, 10) sets the overall figure size in inches.
# This layout allows side-by-side comparison of all key properties.
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Add a main title above all subplots.
# fontsize=16 makes it prominent; fontweight='bold' makes it stand out.
fig.suptitle('Distribution of Molecular Properties', fontsize=16, fontweight='bold')

# Define which DataFrame columns to plot and their human-readable labels.
# These are the six most important drug-likeness descriptors.
props = ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds']
titles = ['Molecular Weight (Da)', 'LogP (Lipophilicity)', 'H-Bond Donors',
          'H-Bond Acceptors', 'Topological PSA', 'Rotatable Bonds']

# Iterate over each subplot axis, property column, and title simultaneously.
# zip() pairs corresponding elements from each list; axes.flat flattens the 2D
# array of axes into a 1D iterator for easy looping.
for ax, prop, title in zip(axes.flat, props, titles):
    # Plot a histogram of the property values.
    # bins=30: divide the data range into 30 equal-width bars.
    # color='#1f77b4': use matplotlib's default blue color.
    # edgecolor='white': white borders between bars for visual separation.
    # alpha=0.7: slight transparency so gridlines show through.
    ax.hist(df[prop], bins=30, color='#1f77b4', edgecolor='white', alpha=0.7)

    # Label the x-axis with the human-readable property name.
    ax.set_xlabel(title)

    # Label the y-axis as "Count" (number of molecules in each bin).
    ax.set_ylabel('Count')

    # Add a vertical dashed red line at the median value.
    # The median is more robust than the mean to outliers.
    # This helps students quickly see the "typical" value.
    ax.axvline(df[prop].median(), color='red', linestyle='--', label=f'Median: {df[prop].median():.1f}')

    # Display the legend showing the median value.
    ax.legend()

# Automatically adjust subplot spacing to prevent overlapping labels.
plt.tight_layout()

# Render and display the figure inline in the notebook.
plt.show()

## 6. Lipinski's Rule of Five

Let's check which molecules in our dataset pass Lipinski's Rule of Five (drug-likeness filter).

### The History
In **1997**, **Christopher A. Lipinski** and colleagues at **Pfizer** published one of the most cited papers in medicinal chemistry. They analyzed the properties of ~2,245 drugs that had reached Phase II clinical trials and observed that poor absorption or permeation was more likely when:

- **MW > 500** Da (molecular weight)
- **LogP > 5** (lipophilicity)
- **HBD > 5** (hydrogen bond donors: NH, OH groups)
- **HBA > 10** (hydrogen bond acceptors: N, O atoms)

The rules are called the "Rule of **Five**" because all thresholds are multiples of 5. Compounds that violate two or more rules are less likely to be orally bioavailable.

### Important caveats
- The Rule of Five applies to **passive transcellular absorption** only — it doesn't apply to compounds that are actively transported across membranes (e.g., many antibiotics, antifungals)
- Natural products frequently violate these rules but are still bioactive drugs
- Antibodies, peptides, and oligonucleotides operate by completely different rules ("beyond Rule of Five" or "bRo5" space)
- ~6% of approved oral drugs violate 2+ rules — so it's a guideline, not an absolute law
- Veber's rules (2002) add two more criteria: TPSA ≤ 140 Å² and rotatable bonds ≤ 10

### Impact on drug discovery
Despite its simplicity, the Rule of Five fundamentally changed how medicinal chemists think about molecular properties. It shifted the field toward **property-based drug design**, where physicochemical properties are optimized alongside potency.

**Reference:** Lipinski, C.A. et al. (1997). Experimental and computational approaches to estimate solubility and permeability in drug discovery and development settings. *Adv. Drug Delivery Rev.* 23:3-25.

In [ ]:
# ============================================================
# LIPINSKI'S RULE OF FIVE — DRUG-LIKENESS FILTER
# Christopher Lipinski at Pfizer published these rules in 1997.
# They predict whether a compound is likely to be orally active
# (absorbed from the gut into the bloodstream) based on simple
# molecular properties. ~90% of approved oral drugs pass these rules.
# The "Rule of Five" name comes from the thresholds being multiples of 5.
# Reference: Lipinski et al. (1997), Adv. Drug Deliv. Rev. 23:3-25
# ============================================================

# Define a function that checks if a molecule passes all four Lipinski rules.
# It takes a pandas DataFrame row (with pre-computed properties) as input
# and returns True if ALL four criteria are met, False otherwise.
# The four rules are:
#   1. MW <= 500 Da (large molecules have poor absorption)
#   2. LogP <= 5 (very lipophilic molecules have poor solubility)
#   3. HBD <= 5 (too many H-bond donors reduce membrane permeation)
#   4. HBA <= 10 (too many H-bond acceptors reduce membrane permeation)
def lipinski_pass(row):
    return (row['MW'] <= 500 and row['LogP'] <= 5 and
            row['HBD'] <= 5 and row['HBA'] <= 10)

# Apply the lipinski_pass function to every row in the DataFrame.
# .apply(..., axis=1) applies the function row-wise (axis=1 means "across columns").
# The result is a new boolean column: True = passes, False = fails.
df['Lipinski_Pass'] = df.apply(lipinski_pass, axis=1)

# Count how many molecules pass the filter.
# .sum() on a boolean Series counts the True values (True=1, False=0).
n_pass = df['Lipinski_Pass'].sum()

# Get the total number of molecules for percentage calculation.
n_total = len(df)

# Print a summary report of pass/fail statistics.
print(f'Lipinski Rule of Five results:')
print(f'  Pass: {n_pass} ({100*n_pass/n_total:.1f}%)')
print(f'  Fail: {n_total - n_pass} ({100*(n_total-n_pass)/n_total:.1f}%)')

In [ ]:
# ============================================================
# VISUALIZE LIPINSKI RULE OF FIVE ANALYSIS
# This creates four histograms, one per Lipinski rule, each with
# a red threshold line showing the cutoff. Molecules to the left
# of the line pass that particular rule.
# ============================================================

# Create a figure with 1 row and 4 columns of subplots (one per rule).
# figsize=(16, 4) makes a wide, short figure suitable for side-by-side comparison.
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Add a main title for the entire figure.
fig.suptitle("Lipinski Rule of Five Analysis", fontsize=14, fontweight='bold')

# Define tuples of (property_column, threshold_value, axis_label) for each rule.
# This data-driven approach avoids repetitive code for each subplot.
rules = [('MW', 500, 'MW <= 500'), ('LogP', 5, 'LogP <= 5'),
         ('HBD', 5, 'HBD <= 5'), ('HBA', 10, 'HBA <= 10')]

# Loop through each subplot axis and its corresponding rule definition.
for ax, (prop, threshold, label) in zip(axes, rules):
    # Plot the histogram of this property across all molecules.
    # bins=30 provides good granularity for visualizing the distribution.
    ax.hist(df[prop], bins=30, color='#1f77b4', edgecolor='white', alpha=0.7)

    # Draw a vertical red dashed line at the Lipinski threshold.
    # Molecules to the LEFT of this line pass this particular rule.
    # linewidth=2 makes the threshold line clearly visible.
    ax.axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold}')

    # Label the x-axis with the rule description (e.g., "MW <= 500").
    ax.set_xlabel(label)

    # Label the y-axis with the count of molecules in each bin.
    ax.set_ylabel('Count')

    # Show the legend with the threshold value.
    ax.legend()

# Adjust spacing between subplots to prevent label overlap.
plt.tight_layout()

# Render the figure inline in the notebook.
plt.show()

## 7. Molecular Similarity

Let's compute Tanimoto similarity between molecules using Morgan fingerprints.

### What are molecular fingerprints?
**Molecular fingerprints** are fixed-length binary (or count) vectors that encode the presence or absence of specific structural features in a molecule. They are the most widely used molecular representation in drug discovery for:
- **Virtual screening:** searching databases of millions of molecules for those similar to a known active
- **Clustering:** grouping molecules by structural similarity
- **SAR analysis:** understanding structure-activity relationships
- **Diversity analysis:** ensuring chemical libraries cover diverse structural space

### Types of fingerprints
| Fingerprint | Description | Use case |
|------------|-------------|----------|
| **Morgan/ECFP** | Circular fingerprint based on atom neighborhoods; most popular | General similarity, ML features |
| **MACCS keys** | 166 pre-defined structural keys (specific functional groups) | Quick structural comparison |
| **RDKit FP** | Topological path-based fingerprint | Alternative to Morgan |
| **Atom Pairs** | Encodes pairs of atoms and their topological distance | Scaffold hopping |

### Morgan fingerprints (ECFP)
Morgan fingerprints — also called **Extended-Connectivity Fingerprints (ECFP)** — were formalized by Rogers & Hahn (2010). They work by:
1. Assigning initial identifiers to each atom (based on atomic number, charge, etc.)
2. Iteratively updating each atom's identifier by hashing it with its neighbors' identifiers
3. Repeating for *radius* iterations (radius=2 → ECFP4, radius=3 → ECFP6)
4. Folding the resulting identifiers into a fixed-length bit vector

### Tanimoto similarity
The **Tanimoto coefficient** (also called Jaccard index) is the standard similarity metric for bit vectors:

**Tanimoto(A, B) = |A ∩ B| / |A ∪ B| = c / (a + b - c)**

where *a* = bits set in A, *b* = bits set in B, *c* = bits set in both A and B.

- **1.0** = identical fingerprints (same structural features)
- **0.0** = completely different (no shared features)
- In drug discovery, Tanimoto > **0.85** often indicates compounds with similar biological activity (but this threshold varies by fingerprint type and target)

**Reference:** Rogers, D. & Hahn, M. (2010). Extended-connectivity fingerprints. *J. Chem. Inf. Model.* 50:742-754.

In [ ]:
# ============================================================
# MOLECULAR SIMILARITY USING MORGAN FINGERPRINTS
# Molecular fingerprints encode structural features as bit vectors.
# Morgan fingerprints (also called circular fingerprints or ECFP)
# capture the local chemical environment around each atom up to a
# given radius. They were developed by Rogers & Hahn (2010),
# J. Chem. Inf. Model. 50:742-754, extending Morgan's original
# algorithm from 1965. ECFP4 (radius=2) is the most commonly used
# fingerprint in pharma for virtual screening and similarity searching.
# ============================================================

# Import AllChem again (it was imported above, but this cell may run standalone).
# AllChem provides the GetMorganFingerprintAsBitVect function we need.
from rdkit.Chem import AllChem

# Import DataStructs which provides similarity metric functions.
# TanimotoSimilarity is the most widely used metric for comparing
# molecular fingerprints — it's the ratio of shared bits to total set bits.
from rdkit import DataStructs

# Select the first 20 molecules from the DataFrame for this analysis.
# We use a subset to keep the similarity matrix a manageable size for visualization.
# In practice, similarity searches are run on millions of molecules.
sample = df.head(20)

# Generate Morgan fingerprints (radius=2, ECFP4-equivalent) for each molecule.
# GetMorganFingerprintAsBitVect() returns a fixed-length bit vector (0s and 1s).
# radius=2: considers atoms up to 2 bonds away from each center atom.
#   This captures functional groups, ring systems, and local topology.
# nBits=2048: the fingerprint length. Longer = fewer hash collisions = more accurate,
#   but uses more memory. 2048 is the standard in most pharma applications.
# Each bit represents a hashed substructural feature; bit=1 means that feature is present.
fps = [AllChem.GetMorganFingerprintAsBitVect(m, radius=2, nBits=2048) for m in sample['mol']]

# Compute the pairwise Tanimoto similarity matrix.
# Tanimoto similarity = |A ∩ B| / |A ∪ B| where A,B are sets of "on" bits.
# Range is 0 (completely different) to 1 (identical fingerprints).
# In drug discovery, molecules with Tanimoto > 0.85 are often considered similar.
# The diagonal is always 1.0 (a molecule is identical to itself).
n = len(fps)

# Initialize an n×n matrix of zeros to store pairwise similarities.
sim_matrix = np.zeros((n, n))

# Compute all pairwise similarities using a nested loop.
# For n=20, this is 400 comparisons — fast. For large datasets,
# you'd use optimized bulk similarity functions.
for i in range(n):
    for j in range(n):
        # DataStructs.TanimotoSimilarity computes the Tanimoto coefficient
        # between two fingerprint bit vectors. This is an O(nBits) operation.
        sim_matrix[i][j] = DataStructs.TanimotoSimilarity(fps[i], fps[j])

# Create a heatmap visualization of the similarity matrix.
# plt.figure(figsize=(10, 8)) creates a new 10x8 inch figure.
plt.figure(figsize=(10, 8))

# sns.heatmap() creates a color-coded matrix visualization.
# cmap='YlOrRd': Yellow-Orange-Red color gradient (low=yellow, high=red).
# vmin=0, vmax=1: fix the color scale to the full Tanimoto range.
# xticklabels/yticklabels: label axes with molecule indices.
sns.heatmap(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1,
            xticklabels=range(n), yticklabels=range(n))

# Add an informative title describing what's being shown.
plt.title('Tanimoto Similarity Matrix (Morgan FP, radius=2)', fontsize=14)

# Label axes to clarify that indices correspond to molecules in the DataFrame.
plt.xlabel('Molecule Index')
plt.ylabel('Molecule Index')

# Adjust layout so labels aren't clipped.
plt.tight_layout()

# Render the heatmap inline in the notebook.
plt.show()

# Calculate and print the average pairwise similarity (excluding self-similarity).
# np.triu_indices(n, k=1) returns indices for the upper triangle (excluding diagonal).
# This avoids double-counting pairs and including the diagonal (always 1.0).
print(f'Average pairwise similarity: {sim_matrix[np.triu_indices(n, k=1)].mean():.3f}')

## Biophysical Modelling: From Neurons to Drug Targets

This section bridges **computational neuroscience** and **drug discovery**, introducing the foundational mathematical models that describe how cells generate electrical signals — and how drugs modify these signals.

### The Hodgkin-Huxley Model (1952)

In 1952, **Alan Hodgkin** and **Andrew Huxley** published a series of five landmark papers describing the mathematical basis of the action potential in the squid giant axon. This work earned them the **Nobel Prize in Physiology or Medicine in 1963** (shared with John Eccles). The Hodgkin-Huxley (HH) model remains the foundation of computational neuroscience and is directly relevant to drug discovery because **~18% of all approved drugs target ion channels**.

### The Equations

The core equation describes how the membrane potential *V* changes over time:

**C_m · dV/dt = −g_Na · m³ · h · (V − E_Na) − g_K · n⁴ · (V − E_K) − g_L · (V − E_L) + I_ext**

Where:
- **C_m** (~1 µF/cm²): Membrane capacitance — the cell membrane acts as a tiny capacitor storing charge
- **g_Na** (120 mS/cm²): Maximum sodium conductance — controls the rapid upstroke of the action potential
- **g_K** (36 mS/cm²): Maximum potassium conductance — controls repolarization
- **g_L** (0.3 mS/cm²): Leak conductance — a small background current
- **E_Na** (+50 mV), **E_K** (−77 mV), **E_L** (−54.4 mV): Reversal (Nernst) potentials determined by ion concentration gradients
- **I_ext**: External stimulation current (e.g., from the SpikerBot or a synaptic input)

### Gating Variables: m, h, and n

The HH model introduces three **gating variables** that describe the state of ion channels:

| Variable | Range | What it represents | Associated channel |
|----------|-------|--------------------|--------------------|
| **m** | 0–1 | Na⁺ channel **activation** — probability that the channel is open | Voltage-gated Na⁺ (Nav) |
| **h** | 0–1 | Na⁺ channel **inactivation** — probability that the channel is NOT inactivated | Voltage-gated Na⁺ (Nav) |
| **n** | 0–1 | K⁺ channel **activation** — probability that the channel is open | Voltage-gated K⁺ (Kv) |

The Na⁺ current depends on **m³·h**: three activation gates AND one inactivation gate must be in the right state. The K⁺ current depends on **n⁴**: four activation gates must be open. Each gating variable follows a first-order ODE: **dx/dt = α_x(V)·(1−x) − β_x(V)·x**, where α and β are voltage-dependent rate constants derived from Hodgkin and Huxley's voltage clamp experiments on the squid giant axon.

### Why This Matters for Drug Discovery

Drugs that target ion channels represent a massive pharmacological category:

| Drug | Target | Therapeutic Use | Mechanism |
|------|--------|-----------------|-----------|
| **Lidocaine** | Nav1.7 (Na⁺ channel) | Local anesthetic | Blocks Na⁺ channel → reduces g_Na → prevents action potentials in pain neurons |
| **Carbamazepine** | Nav channels | Epilepsy, neuropathic pain | Stabilizes inactivated state of Na⁺ channels |
| **Dofetilide** | hERG/Kv11.1 (K⁺ channel) | Cardiac arrhythmia | Blocks K⁺ channel → reduces g_K → prolongs cardiac action potential |
| **Amlodipine** | Cav1.2 (Ca²⁺ channel) | Hypertension | Blocks L-type Ca²⁺ channels in vascular smooth muscle |
| **Diazepam** | GABA_A receptor | Anxiety, seizures | Enhances Cl⁻ channel opening → increases inhibitory conductance |
| **Gabapentin** | α2δ subunit of Cav | Neuropathic pain | Modulates Ca²⁺ channel trafficking |

**⚠️ The hERG safety problem:** The hERG (human Ether-à-go-go-Related Gene) potassium channel (Kv11.1) is critical for cardiac repolarization. Many drugs unintentionally block hERG, causing **QT prolongation** on the ECG, which can lead to fatal cardiac arrhythmias (Torsades de Pointes). hERG liability screening is **mandatory** in drug development — multiple drugs (terfenadine, cisapride, rofecoxib) have been withdrawn from the market due to hERG-related cardiac toxicity.

### The SpikerBot (Backyard Brains)

The **SpikerBot** is an educational neuroscience tool developed by **Backyard Brains** (Ann Arbor, Michigan), a company founded by **Tim Marzullo** and **Greg Gage** to bring neuroscience experiments to students at all levels.

**What the SpikerBot does:**
- Generates **programmable neural motifs** — electrical patterns that mimic real neuron firing patterns
- Can produce **tonic firing** (regular, metronome-like spikes), **burst firing** (groups of rapid spikes separated by pauses), and **irregular firing** (stochastic patterns)
- Connects to cockroach leg preparations, allowing students to see how electrical stimulation produces muscle contractions — the same principle underlying deep brain stimulation (DBS) for Parkinson's disease and cardiac pacemakers
- Demonstrates the same biophysical principles captured by the Hodgkin-Huxley model

**The connection to drug discovery:** The SpikerBot's electrical patterns arise from the same ion channel dynamics described by the HH model. When a drug modifies ion channel conductance (g_Na, g_K, etc.), it changes the neuron's firing pattern — this is how anesthetics silence pain neurons, how anti-epileptics prevent seizure activity, and how cardiac drugs regulate heart rhythm.

### References
- Hodgkin, A.L. & Huxley, A.F. (1952). A quantitative description of membrane current and its application to conduction and excitation in nerve. *J. Physiol.* 117:500-544
- Marzullo, T.C. & Gage, G.J. (2012). The SpikerBox: A Low Cost, Open-Source BioAmplifier for Increasing Public Participation in Neuroscience Inquiry. *Adv. Physiol. Educ.* 36:2-14
- Hille, B. (2001). *Ion Channels of Excitable Membranes* (3rd ed.). Sinauer Associates.
- Sanguinetti, M.C. & Tristani-Firouzi, M. (2006). hERG potassium channels and cardiac arrhythmia. *Nature* 440:463-469

### Dr. Serbe-Kamp’s Research: GluClα Channels and Visual Motion Detection

Dr. Étienne Serbe-Kamp’s work on *Drosophila* visual motion circuits provides a vivid example of how ion-channel physiology connects to neural computation — and, ultimately, to drug targets. In *Multilevel visual motion opponency in Drosophila* ([Nature Neuroscience, 2023](https://doi.org/10.1038/s41593-023-01443-z)), his team showed that **GluClα (glutamate-gated chloride channel alpha)** mediates an inhibitory conductance that is essential for direction-selective motion opponency. GluClα belongs to the same channel superfamily targeted by **ivermectin**, one of the most successful antiparasitic drugs in history (Nobel Prize 2015, Campbell & Ōmura). When ivermectin binds GluClα, it locks the channel open, producing a sustained Cl⁻ conductance that hyperpolarises the cell and silences neural activity — exactly the mechanism we simulate below.

### The *Drosophila* Connectome: A Computational Neuroscience Resource

Serbe-Kamp’s earlier work ([Shinomiya, Serbe-Kamp et al., *Neuron* 2016/2019](https://doi.org/10.1016/j.neuron.2019.02.043)) used **serial-section electron microscopy (EM) connectomics** to comprehensively map every presynaptic input to the T5 motion-detector neurons. The *Drosophila* connectome — a complete wiring diagram of the fly brain — is now one of the most powerful resources in computational neuroscience. It allows researchers to build biologically constrained circuit models (like the HH model below, but with realistic connectivity) and to identify exactly which synaptic receptors (GluClα, Rdl/GABA-A, nAChR) mediate each connection. For drug discovery, connectomics reveals which receptor subtypes are expressed at which synapses, guiding the design of subtype-selective compounds.

### The Cys-Loop Receptor Superfamily: A Major Drug-Target Family

GluClα belongs to the **Cys-loop ligand-gated ion channel superfamily**, one of the most pharmacologically important protein families in all of biology. Members share a conserved extracellular disulfide bond (the “Cys-loop”) and form pentameric channels:

| Receptor | Endogenous ligand | Ion selectivity | Key drugs | Clinical use |
|----------|-------------------|-----------------|-----------|-------------|
| **GABA-A** | GABA | Cl⁻ (inhibitory) | Diazepam, phenobarbital, propofol | Anxiety, epilepsy, anaesthesia |
| **GluCl** | Glutamate | Cl⁻ (inhibitory) | Ivermectin | Antiparasitic (river blindness, scabies) |
| **nAChR** | Acetylcholine | Na⁺/K⁺ (excitatory) | Nicotine, varenicline | Smoking cessation, myasthenia gravis |
| **Glycine receptor** | Glycine | Cl⁻ (inhibitory) | Strychnine (antagonist) | — (research tool) |
| **5-HT₃** | Serotonin | Na⁺/K⁺ (excitatory) | Ondansetron | Chemotherapy-induced nausea |

All five receptor types can be modelled using the same Hodgkin-Huxley formalism: each contributes a conductance term **g · (V − E_rev)** to the membrane equation, where **E_rev** depends on which ions permeate the channel. For Cl⁻-selective channels (GABA-A, GluCl, glycine), E_rev ≈ −70 to −80 mV (close to or below resting potential), producing **inhibition** — the neuron is clamped near rest and cannot fire. This is exactly the mechanism we will simulate in the GluCl/ivermectin extension below.

### Additional References
- Serbe-Kamp, E. et al. (2023). Multilevel visual motion opponency in Drosophila. *Nature Neuroscience* 26:1894–1905
- Shinomiya, K., Serbe-Kamp, E. et al. (2019). Comprehensive characterization of presynaptic elements to identified T5 neurons. *Neuron* 104(2):P307-320.E5
- Serbe, E. et al. (2016). Comprehensive characterization of the major presynaptic elements to the Drosophila OFF motion detector. *Neuron* 89(4):829-841
- Cully, D.F. et al. (1994). Cloning of an avermectin-sensitive glutamate-gated chloride channel from *Caenorhabditis elegans*. *Nature* 371:707-711
- Lynagh, T. & Lynch, J.W. (2012). Ivermectin binding sites in human and invertebrate Cys-loop receptors. *Trends Pharmacol. Sci.* 33:432-441

In [ ]:
# ============================================================
# HODGKIN-HUXLEY ACTION POTENTIAL SIMULATION
# This simulates the electrical behavior of a neuron using the
# Nobel Prize-winning model from 1952. Understanding this model
# is crucial for drug discovery because ~18% of approved drugs
# target the ion channels described by these equations.
# Reference: Hodgkin & Huxley (1952), J. Physiol. 117:500-544
# ============================================================

# Import NumPy for numerical array operations and mathematical functions.
# We need arrays to store the time-series of voltage and gating variables,
# and functions like np.exp() for the exponential terms in the rate equations.
import numpy as np

# Import matplotlib for creating publication-quality plots.
# We'll create a 3-panel figure showing voltage, gating variables, and stimulus.
import matplotlib.pyplot as plt

# --- Hodgkin-Huxley parameters ---
# C_m: membrane capacitance in microfarads per cm^2
# This represents the ability of the cell membrane to store charge,
# like a tiny biological capacitor. Typical value is ~1 uF/cm^2.
# The lipid bilayer is ~5nm thick, creating a capacitor with this value.
C_m = 1.0

# g_Na: maximum sodium conductance (millisiemens per cm^2)
# This is how easily Na+ ions can flow when ALL sodium channels are open.
# Na+ influx causes the rapid upstroke (depolarization) of the action potential.
# Drugs like lidocaine (local anesthetic) REDUCE this value, blocking pain signals.
g_Na = 120.0

# g_K: maximum potassium conductance (mS/cm^2)
# K+ efflux causes repolarization (return to resting potential).
# The hERG channel (Kv11.1) is a K+ channel. Blocking it causes
# QT prolongation on the ECG -> cardiac arrhythmia -> drug withdrawal!
# This is the #1 safety concern in drug development.
g_K = 36.0

# g_L: leak conductance (mS/cm^2)
# Small background conductance that keeps the neuron near resting potential.
# Represents channels that are always partially open.
g_L = 0.3

# E_Na, E_K, E_L: reversal (Nernst) potentials in millivolts
# These are the voltages at which net ion flow is zero for each ion type.
# They depend on ion concentration gradients across the membrane.
# E_Na is positive because Na+ concentration is higher outside the cell.
# E_K is negative because K+ concentration is higher inside the cell.
E_Na = 50.0    # Sodium reversal potential (mV)
E_K = -77.0    # Potassium reversal potential (mV)
E_L = -54.387  # Leak reversal potential (mV)

# --- Gating variable rate functions ---
# These functions describe how fast channels open and close
# as a function of membrane voltage V.
# alpha = opening rate, beta = closing rate
# m = sodium activation (how quickly Na+ channels open)
# h = sodium inactivation (how quickly Na+ channels close/inactivate)
# n = potassium activation (how quickly K+ channels open)

def alpha_m(V):
    # Rate of Na+ channel activation gate opening.
    # This increases steeply with depolarization (more positive V),
    # which is why Na+ channels open rapidly during the action potential upstroke.
    # The formula was empirically fit to voltage clamp data from the squid giant axon.
    return 0.1 * (V + 40.0) / (1.0 - np.exp(-(V + 40.0) / 10.0))

def beta_m(V):
    # Rate of Na+ channel activation gate closing.
    # This is high at resting potential, keeping Na+ channels closed at rest.
    return 4.0 * np.exp(-(V + 65.0) / 18.0)

def alpha_h(V):
    # Rate of Na+ channel inactivation gate recovery.
    # When h is low, the Na+ channel is inactivated (blocked from inside).
    # alpha_h promotes recovery from inactivation (h increases toward 1).
    return 0.07 * np.exp(-(V + 65.0) / 20.0)

def beta_h(V):
    # Rate of Na+ channel inactivation gate closing.
    # h goes from 1 (channel available) to 0 (channel inactivated).
    # At depolarized voltages, beta_h is high -> h decreases -> channels inactivate.
    # This inactivation is what terminates the Na+ current during an action potential.
    return 1.0 / (1.0 + np.exp(-(V + 35.0) / 10.0))

def alpha_n(V):
    # Rate of K+ channel activation gate opening.
    # K+ channels open more slowly than Na+ channels — this delay is what
    # allows the action potential: Na+ rushes in first, then K+ flows out.
    return 0.01 * (V + 55.0) / (1.0 - np.exp(-(V + 55.0) / 10.0))

def beta_n(V):
    # Rate of K+ channel activation gate closing.
    # At resting potential, n is low (K+ channels mostly closed).
    return 0.125 * np.exp(-(V + 65.0) / 80.0)

# --- Simulation setup ---
# dt: time step in milliseconds. Smaller = more accurate but slower.
# Using Euler integration, dt must be small enough for numerical stability.
# 0.01 ms is a good balance of accuracy and speed for the HH model.
dt = 0.01   # Time step (ms) - 0.01 ms gives good accuracy

# T: total simulation time in milliseconds.
# 50 ms is enough to see several action potentials with sustained stimulation.
T = 50.0    # Total simulation time (ms) - enough to see several spikes

# Create time array from 0 to T with step dt.
# np.arange creates evenly spaced values: [0, 0.01, 0.02, ..., 50.0]
# This will have 5000 time points.
t = np.arange(0, T, dt)

# Initialize arrays to store results over time.
# np.zeros creates an array of zeros with the same length as t.
# We'll fill these in during the simulation loop.
V = np.zeros(len(t))   # Membrane potential (mV) over time
m = np.zeros(len(t))   # Na+ activation gating variable (0 to 1)
h = np.zeros(len(t))   # Na+ inactivation gating variable (0 to 1)
n = np.zeros(len(t))   # K+ activation gating variable (0 to 1)

# Set initial conditions at t=0.
# V starts at the resting potential (-65 mV), which is close to E_K
# because at rest, K+ channels dominate membrane permeability.
V[0] = -65.0

# Gating variables start at their steady-state values at resting potential.
# At steady state: dx/dt = 0, so x_inf = alpha / (alpha + beta).
# This ensures the simulation starts in a biologically realistic state.
m[0] = alpha_m(V[0]) / (alpha_m(V[0]) + beta_m(V[0]))
h[0] = alpha_h(V[0]) / (alpha_h(V[0]) + beta_h(V[0]))
n[0] = alpha_n(V[0]) / (alpha_n(V[0]) + beta_n(V[0]))

# External stimulation current (microamps per cm^2).
# This is like the electrical pulse from the SpikerBot!
# We create an array of zeros and then set a sustained current pulse.
I_ext = np.zeros(len(t))

# Inject 10 uA/cm^2 from t=5ms to t=45ms to trigger action potentials.
# The current must exceed a threshold (~6-7 uA/cm^2 for these parameters)
# to depolarize the membrane enough to trigger an action potential.
# Below threshold: subthreshold oscillations only. Above: repetitive firing.
I_ext[(t >= 5) & (t <= 45)] = 10.0  # 10 uA/cm^2 sustained current

# --- Euler integration of Hodgkin-Huxley equations ---
# We step through time, computing the next state from the current state.
# Euler's method: x(t+dt) = x(t) + dt * dx/dt
# This is the simplest numerical integration method. More sophisticated
# methods (Runge-Kutta 4) are more accurate but Euler suffices here.
for i in range(len(t) - 1):
    # Compute ionic currents at current time step.
    # Each current = conductance * gating * driving force.
    # The driving force (V - E_ion) determines the direction and magnitude
    # of ion flow: positive = outward current, negative = inward current.

    # Sodium current: g_Na * m^3 * h * (V - E_Na)
    # m^3 means three activation gates must be open simultaneously.
    # h means the inactivation gate must NOT be closed.
    # At the peak of the action potential, m is high and h hasn't dropped yet.
    I_Na = g_Na * m[i]**3 * h[i] * (V[i] - E_Na)  # Sodium current

    # Potassium current: g_K * n^4 * (V - E_K)
    # n^4 means four activation gates must be open.
    # K+ channels open more slowly than Na+ channels, causing repolarization
    # AFTER the Na+ channels have already started to inactivate.
    I_K = g_K * n[i]**4 * (V[i] - E_K)              # Potassium current

    # Leak current: g_L * (V - E_L)
    # Always-on background current that stabilizes the resting potential.
    I_L = g_L * (V[i] - E_L)                         # Leak current

    # Update membrane potential using Euler method.
    # C_m * dV/dt = I_ext - I_Na - I_K - I_L
    # Rearranging: dV/dt = (I_ext - I_Na - I_K - I_L) / C_m
    V[i+1] = V[i] + dt * (I_ext[i] - I_Na - I_K - I_L) / C_m

    # Update gating variables using Euler method.
    # General form: dx/dt = alpha_x(V) * (1-x) - beta_x(V) * x
    # alpha*(1-x) is the rate of closed->open transitions.
    # beta*x is the rate of open->closed transitions.
    m[i+1] = m[i] + dt * (alpha_m(V[i]) * (1-m[i]) - beta_m(V[i]) * m[i])
    h[i+1] = h[i] + dt * (alpha_h(V[i]) * (1-h[i]) - beta_h(V[i]) * h[i])
    n[i+1] = n[i] + dt * (alpha_n(V[i]) * (1-n[i]) - beta_n(V[i]) * n[i])

# --- Plot the results ---
# Create a figure with 3 vertically stacked subplots sharing the x-axis (time).
# figsize=(12, 10) makes the figure large enough to see details clearly.
# sharex=True links the x-axes so zooming on one panel zooms all panels.
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Top panel: membrane potential over time (the action potentials!).
# This is the primary output of the HH model — what an oscilloscope or
# SpikerBot would actually record from a real neuron.
axes[0].plot(t, V, 'b-', linewidth=1.5)

# Label the y-axis with the physical quantity and units.
axes[0].set_ylabel('Membrane Potential (mV)', fontsize=12)

# Add a descriptive title referencing the SpikerBot connection.
axes[0].set_title('Hodgkin-Huxley Action Potential Simulation\n'
                   'This is what the SpikerBot records from real neurons!', fontsize=14)

# Draw a horizontal line at the approximate firing threshold (~-55 mV).
# When V crosses this threshold, the positive feedback loop of Na+ channel
# opening triggers a full action potential (all-or-none response).
axes[0].axhline(y=-55, color='r', linestyle='--', alpha=0.5, label='Threshold (~-55 mV)')

# Display the legend showing the threshold line label.
axes[0].legend(fontsize=10)

# Middle panel: gating variables (channel states) over time.
# This shows the molecular-level dynamics of ion channels that produce
# the action potential. These are the drug targets!
axes[1].plot(t, m, 'r-', label='m (Na+ activation)', linewidth=1.5)
axes[1].plot(t, h, 'g-', label='h (Na+ inactivation)', linewidth=1.5)
axes[1].plot(t, n, 'b-', label='n (K+ activation)', linewidth=1.5)

# Label y-axis — gating variables range from 0 (fully closed) to 1 (fully open).
axes[1].set_ylabel('Gating Variable (0-1)', fontsize=12)

# Title emphasizing the pharmacological relevance.
axes[1].set_title('Ion Channel Gating Variables - Drug targets!', fontsize=14)

# Show legend so students can identify which trace is which.
axes[1].legend(fontsize=10)

# Bottom panel: external stimulation current over time.
# This shows when and how much current we injected into the model neuron.
axes[2].plot(t, I_ext, 'k-', linewidth=1.5)

# Label y-axis with the physical quantity and units.
# The µ symbol is a Unicode character for micro.
axes[2].set_ylabel('Stimulus Current (\u00b5A/cm\u00b2)', fontsize=12)

# Label x-axis (shared across all panels due to sharex=True).
axes[2].set_xlabel('Time (ms)', fontsize=12)

# Title connecting to the SpikerBot educational tool.
axes[2].set_title('External Stimulation (like SpikerBot pulse)', fontsize=14)

# Automatically adjust subplot spacing to prevent overlapping labels.
plt.tight_layout()

# Render the figure inline in the notebook.
plt.show()

# Print an educational summary of the simulation results.
# This helps students interpret what they see in the plots.
print("=== WHAT YOU JUST SIMULATED ===")

# Count action potentials by detecting positive-going zero crossings.
# np.diff detects changes in the boolean (V > 0) signal.
# Each transition from False to True (0 to 1) is one action potential.
print(f"Number of action potentials: ~{np.sum(np.diff((V > 0).astype(int)) == 1)}")

# Report the resting potential (initial condition).
print(f"Resting potential: {V[0]:.1f} mV")

# Report the peak membrane potential reached during the simulation.
print(f"Peak potential: {V.max():.1f} mV")

# Explain what drug effects could be simulated by changing parameters.
# This connects the biophysics back to drug discovery applications.
print("\nDrug effects you could simulate:")
print("  - Reduce g_Na (lidocaine): fewer/smaller action potentials")
print("  - Reduce g_K (hERG block): prolonged repolarization (QT prolongation!)")
print("  - Increase g_K: shortened action potentials")


# ============================================================
# EXTENSION: SIMULATING GluCl CHANNEL ACTIVATION BY IVERMECTIN
# ============================================================
#
# BACKGROUND
# ----------
# GluCl-alpha (glutamate-gated chloride channel alpha) is a Cl-selective
# ligand-gated ion channel belonging to the Cys-loop receptor superfamily.
# In Drosophila, GluCl-alpha mediates inhibitory signalling in visual motion
# circuits (Serbe-Kamp et al., Nature Neuroscience 2023).
#
# IVERMECTIN MECHANISM
# --------------------
# Ivermectin (a macrocyclic lactone, Nobel Prize 2015) is a potent
# and essentially irreversible agonist/positive allosteric modulator
# of GluCl channels. It binds at the transmembrane domain interface
# between subunits, locking the channel in its open state. This:
#   1. Increases Cl- conductance (g_GluCl goes UP)
#   2. Drives the membrane potential toward E_Cl (~-80 mV)
#   3. Hyperpolarises the cell, preventing action potential firing
#   4. In parasites: paralysis and death (motor neurons silenced)
#   5. In Drosophila vision: disrupts direction-selective inhibition
#
# In the HH formalism, we add a GluCl conductance term:
#   I_GluCl = g_GluCl * (V - E_Cl)
# where E_Cl ~ -80 mV (Cl- reversal potential) and g_GluCl is the
# conductance that increases when ivermectin is applied.
# ============================================================

# --- GluCl / Ivermectin parameters ---
# E_Cl: chloride reversal potential (mV).
# In most neurons, intracellular Cl- is low (~5 mM) vs extracellular
# (~110 mM), giving E_Cl around -80 mV. Since E_Cl is below the
# resting potential (~-65 mV), opening Cl- channels HYPERPOLARISES
# the cell, making it harder to fire.
E_Cl = -80.0

# g_GluCl: GluCl conductance added by ivermectin (mS/cm^2).
# We compare three conditions:
#   0.0 = no drug (control)
#   1.0 = moderate ivermectin exposure (partial channel activation)
#   5.0 = high ivermectin exposure (channels locked open, strong inhibition)
# For reference, the HH leak conductance is 0.3 mS/cm^2, so even
# g_GluCl = 1.0 is a significant additional inhibitory conductance.
glucl_conductances = [0.0, 1.0, 5.0]
labels_glucl = ['Control (no drug)', 'Ivermectin (low: g_GluCl=1.0)',
                'Ivermectin (high: g_GluCl=5.0)']
colors_glucl = ['blue', 'orange', 'red']

fig_glucl, axes_glucl = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

for idx, g_GluCl in enumerate(glucl_conductances):
    # Re-initialise state variables for each condition
    V_glucl = np.zeros(len(t))
    m_glucl = np.zeros(len(t))
    h_glucl = np.zeros(len(t))
    n_glucl = np.zeros(len(t))

    V_glucl[0] = -65.0
    m_glucl[0] = alpha_m(V_glucl[0]) / (alpha_m(V_glucl[0]) + beta_m(V_glucl[0]))
    h_glucl[0] = alpha_h(V_glucl[0]) / (alpha_h(V_glucl[0]) + beta_h(V_glucl[0]))
    n_glucl[0] = alpha_n(V_glucl[0]) / (alpha_n(V_glucl[0]) + beta_n(V_glucl[0]))

    # Euler integration with the additional GluCl Cl- conductance.
    # The only change vs the standard HH model is the I_GluCl term.
    for i in range(len(t) - 1):
        I_Na_g = g_Na * m_glucl[i]**3 * h_glucl[i] * (V_glucl[i] - E_Na)
        I_K_g  = g_K  * n_glucl[i]**4 * (V_glucl[i] - E_K)
        I_L_g  = g_L  * (V_glucl[i] - E_L)

        # GluCl current: Cl- flows inward (hyperpolarising) when V > E_Cl.
        # Ivermectin holds the channel open, so this conductance is always on
        # (no gating variable needed -- the drug locks the gate open).
        I_GluCl = g_GluCl * (V_glucl[i] - E_Cl)

        # Membrane equation now includes I_GluCl as an additional current
        V_glucl[i+1] = V_glucl[i] + dt * (
            I_ext[i] - I_Na_g - I_K_g - I_L_g - I_GluCl
        ) / C_m

        m_glucl[i+1] = m_glucl[i] + dt * (alpha_m(V_glucl[i]) * (1-m_glucl[i]) - beta_m(V_glucl[i]) * m_glucl[i])
        h_glucl[i+1] = h_glucl[i] + dt * (alpha_h(V_glucl[i]) * (1-h_glucl[i]) - beta_h(V_glucl[i]) * h_glucl[i])
        n_glucl[i+1] = n_glucl[i] + dt * (alpha_n(V_glucl[i]) * (1-n_glucl[i]) - beta_n(V_glucl[i]) * n_glucl[i])

    # Plot membrane potential for this condition
    axes_glucl[0].plot(t, V_glucl, color=colors_glucl[idx],
                       label=labels_glucl[idx], linewidth=1.5)

    # Count spikes for the summary panel
    spike_count = np.sum(np.diff((V_glucl > 0).astype(int)) == 1)
    axes_glucl[1].bar(idx, spike_count, color=colors_glucl[idx],
                      label=labels_glucl[idx])

# --- Format the membrane-potential panel ---
axes_glucl[0].set_ylabel('Membrane Potential (mV)', fontsize=12)
axes_glucl[0].set_title(
    'Effect of GluCl Channel Activation (Ivermectin) on Neuronal Firing\n'
    'Modelling the mechanism studied in Drosophila by Serbe-Kamp et al. (2023)',
    fontsize=13)
axes_glucl[0].axhline(y=-55, color='gray', linestyle='--', alpha=0.4,
                       label='Threshold (~-55 mV)')
axes_glucl[0].legend(fontsize=9, loc='upper right')
axes_glucl[0].set_xlabel('Time (ms)', fontsize=12)

# --- Format the spike-count panel ---
axes_glucl[1].set_ylabel('Number of Action Potentials', fontsize=12)
axes_glucl[1].set_xlabel('Condition', fontsize=12)
axes_glucl[1].set_xticks(range(len(labels_glucl)))
axes_glucl[1].set_xticklabels(['Control', 'Low IVM', 'High IVM'], fontsize=11)
axes_glucl[1].set_title('Ivermectin Dose-Dependently Suppresses Firing', fontsize=13)

plt.tight_layout()
plt.show()

# --- Educational summary ---
print('=== GluCl / IVERMECTIN SIMULATION SUMMARY ===')
print('Ivermectin activates GluCl-alpha channels, adding a Cl- conductance')
print('that clamps the membrane near E_Cl = -80 mV (hyperpolarisation).')
print('This is the same mechanism that:')
print('  1. Kills parasitic worms (river blindness) -- Nobel Prize 2015')
print('  2. Mediates inhibitory signalling in Drosophila motion circuits')
print('     (Serbe-Kamp et al., Nature Neuroscience 2023)')
print('  3. Is shared by all Cys-loop Cl- channels (GABA-A, GluCl, GlyR)')
print()
print('Try modifying g_GluCl values or E_Cl to explore:')
print('  - What happens with very large g_GluCl? (complete silencing)')
print('  - What if E_Cl = -65 mV? (shunting inhibition, not hyperpolarising)')
print('  - What if you also reduce g_Na (combining two drugs)?')


## 8. Exercises

1. **Try different molecules**: Pick 3 drugs you know, find their SMILES on PubChem, and visualize them with RDKit
2. **Property comparison**: Calculate properties for your chosen drugs. Do they pass Lipinski's Rule of Five?
3. **Similarity search**: Which two molecules in the dataset are most similar? Which are most different?
4. **Challenge**: Can you find a molecule in the dataset that violates Lipinski's rules but is still a real drug?

## References
- Weininger, D. (1988). SMILES, a chemical language and information system. J. Chem. Inf. Comput. Sci. 28:31-36
- Lipinski, C.A. et al. (1997). Experimental and computational approaches to estimate solubility and permeability. Adv. Drug Delivery Rev. 23:3-25
- Gaulton, A. et al. (2017). The ChEMBL database in 2017. Nucleic Acids Research 45:D986-D994
- RDKit Documentation: https://www.rdkit.org/docs/